# Multi-Fidelity Optimization Benchmark

This notebook benchmarks fixed- and adaptive-`ksp_rtol` schedules for the inverse solve. The default benchmark records TAO iteration traces, total forward/adjoint KSP work, final objective value, and relative conductivity error.

## Setup

The benchmark helper lives in `src/benchmark_utils.py`. The path setup below keeps the notebook runnable even if the editable install has not been refreshed yet.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CACHE_DIR = PROJECT_ROOT / ".cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR))
os.environ.setdefault("MPLCONFIGDIR", str(CACHE_DIR / "matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PROJECT_ROOT

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from benchmark_utils import (
    benchmark_summaries,
    benchmark_trace_rows,
    default_benchmark_cases,
    default_benchmark_methods,
    run_benchmark_suite,
)

plt.style.use("tableau-colorblind10")
pd.set_option("display.max_columns", 100)


## Benchmark Configuration

The default suite covers three synthetic conductivity fields and five optimization methods. Start with `nmesh=16` and `mit=25` for a quick pass, then scale the mesh and iteration cap once the methodology is stable.

In [ ]:
cases = default_benchmark_cases(nmesh=16, sigma=5e-3, alpha=1e-4)
methods = default_benchmark_methods(mit=25)
reference_method = "fixed-rtol-1e-08"
target_factor = 1.05

print("Cases:", [case.name for case in cases])
print("Methods:", [method.name for method in methods])
print("Reference method:", reference_method)
print("Objective target factor:", target_factor)

## Run The Suite

Each result contains a run summary and a TAO trace. The trace includes the objective, gradient norm, accepted step length, current forward/adjoint `ksp_rtol`, and the KSP iterations spent in the current forward and adjoint solves.

In [ ]:
results = run_benchmark_suite(cases, methods)

summary_df = pd.DataFrame(benchmark_summaries(results)).sort_values(
    ["case_name", "method_name"]
).reset_index(drop=True)
trace_df = pd.DataFrame(benchmark_trace_rows(results)).sort_values(
    ["case_name", "method_name", "tao_iter"]
).reset_index(drop=True)

summary_df

## Compare Against A High-Fidelity Reference

Use the tightest fixed-tolerance method as the reference for each case. The table below converts raw runtime and KSP work into ratios against that reference and computes the final objective gap in percent.

In [ ]:
reference_df = (
    summary_df.loc[
        summary_df["method_name"] == reference_method,
        ["case_name", "final_objective", "wall_time_s", "total_ksp_iterations", "h_error_l2_rel"],
    ]
    .rename(
        columns={
            "final_objective": "reference_objective",
            "wall_time_s": "reference_wall_time_s",
            "total_ksp_iterations": "reference_total_ksp_iterations",
            "h_error_l2_rel": "reference_h_error_l2_rel",
        }
    )
)

summary_with_ref = summary_df.merge(reference_df, on="case_name", how="left")
summary_with_ref["objective_gap_pct"] = 100.0 * (
    summary_with_ref["final_objective"] - summary_with_ref["reference_objective"]
) / summary_with_ref["reference_objective"]
summary_with_ref["wall_time_ratio_vs_reference"] = (
    summary_with_ref["wall_time_s"] / summary_with_ref["reference_wall_time_s"]
)
summary_with_ref["ksp_work_ratio_vs_reference"] = (
    summary_with_ref["total_ksp_iterations"] / summary_with_ref["reference_total_ksp_iterations"]
)
summary_with_ref["h_error_ratio_vs_reference"] = (
    summary_with_ref["h_error_l2_rel"] / summary_with_ref["reference_h_error_l2_rel"]
)

display_columns = [
    "case_name",
    "method_name",
    "wall_time_s",
    "wall_time_ratio_vs_reference",
    "total_ksp_iterations",
    "ksp_work_ratio_vs_reference",
    "final_objective",
    "objective_gap_pct",
    "h_error_l2_rel",
    "h_error_ratio_vs_reference",
    "ksp_tightenings",
]
summary_with_ref[display_columns].round(4)

## Time-To-Target

The trace already contains the objective at every TAO iteration, so we can define a case-specific target as a small multiplicative band above the reference objective and ask how quickly each method reaches it.

In [ ]:
target_df = reference_df[["case_name", "reference_objective"]].copy()
target_df["target_objective"] = target_factor * target_df["reference_objective"]
trace_with_target = trace_df.merge(
    target_df[["case_name", "target_objective"]], on="case_name", how="left"
)

time_to_target_rows = []
for (case_name, method_name), group in trace_with_target.groupby(
    ["case_name", "method_name"], sort=True
):
    group = group.sort_values("tao_iter")
    hits = group.loc[group["objective"] <= group["target_objective"].iloc[0]]
    time_to_target_rows.append(
        {
            "case_name": case_name,
            "method_name": method_name,
            "time_to_target_s": np.nan if hits.empty else float(hits.iloc[0]["wall_time_s"]),
            "iters_to_target": np.nan if hits.empty else int(hits.iloc[0]["tao_iter"]),
        }
    )

time_to_target_df = pd.DataFrame(time_to_target_rows).sort_values(
    ["case_name", "time_to_target_s", "method_name"]
).reset_index(drop=True)
time_to_target_df.round(4)

## Convergence And Tolerance Schedules

In [ ]:
fig, axes = plt.subplots(len(cases), 2, figsize=(13, 4 * len(cases)), squeeze=False)

for row, case in enumerate(cases):
    case_trace = trace_df.loc[trace_df["case_name"] == case.name]
    ax_obj, ax_rtol = axes[row]

    for method_name, group in case_trace.groupby("method_name", sort=False):
        group = group.sort_values("tao_iter")
        ax_obj.plot(
            group["tao_iter"],
            group["objective"],
            marker="o",
            markersize=3,
            label=method_name,
        )
        ax_rtol.semilogy(
            group["tao_iter"],
            group["forward_ksp_rtol"],
            marker="o",
            markersize=3,
            label=method_name,
        )

    ax_obj.set_title(f"{case.name}: objective trace")
    ax_obj.set_xlabel("TAO iteration")
    ax_obj.set_ylabel("J")
    ax_obj.grid(alpha=0.3)

    ax_rtol.set_title(f"{case.name}: forward/adjoint ksp_rtol")
    ax_rtol.set_xlabel("TAO iteration")
    ax_rtol.set_ylabel("ksp_rtol")
    ax_rtol.grid(alpha=0.3)

axes[0, 0].legend(loc="best", fontsize=8)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for case_name, group in summary_with_ref.groupby("case_name"):
    axes[0].scatter(
        group["total_ksp_iterations"],
        group["final_objective"],
        s=80,
        label=case_name,
    )
    for _, row in group.iterrows():
        axes[0].annotate(
            row["method_name"].replace("fixed-rtol-", "F=").replace("adaptive-rtol-", "A="),
            (row["total_ksp_iterations"], row["final_objective"]),
            fontsize=8,
            xytext=(4, 4),
            textcoords="offset points",
        )

axes[0].set_xlabel("Total forward + adjoint KSP iterations")
axes[0].set_ylabel("Final objective")
axes[0].set_title("Work vs. final objective")
axes[0].grid(alpha=0.3)
axes[0].legend(loc="best", fontsize=8)

pivot_time = time_to_target_df.pivot(
    index="method_name", columns="case_name", values="time_to_target_s"
)
pivot_time.plot(kind="bar", ax=axes[1])
axes[1].set_ylabel(f"Wall time to reach {target_factor:.0%} of reference objective [s]")
axes[1].set_title("Time-to-target by method")
axes[1].grid(axis="y", alpha=0.3)
axes[1].legend(title="Case", fontsize=8)
fig.tight_layout()

## Persist The Results

Writing the summary and trace tables to disk makes it easier to compare multiple benchmark runs, meshes, or hardware configurations.

In [ ]:
output_dir = PROJECT_ROOT / "benchmark_output"
output_dir.mkdir(parents=True, exist_ok=True)

summary_path = output_dir / "multifidelity_summary.csv"
trace_path = output_dir / "multifidelity_trace.csv"

summary_with_ref.to_csv(summary_path, index=False)
trace_df.to_csv(trace_path, index=False)

summary_path, trace_path